In [2]:
import torch
import torch.nn as nn

In [3]:
sentences = [
    ["the", "cat", "sat"],
    ["dogs", "are", "playing"],
    ["the", "dog", "is", "running"]
]

In [4]:
words = sorted(list(set([word for sentence in sentences for word in sentence])))

In [5]:
words

['are', 'cat', 'dog', 'dogs', 'is', 'playing', 'running', 'sat', 'the']

In [6]:
word_to_idx = {word: i+1 for i, word in enumerate(words)}

In [7]:
word_to_idx

{'are': 1,
 'cat': 2,
 'dog': 3,
 'dogs': 4,
 'is': 5,
 'playing': 6,
 'running': 7,
 'sat': 8,
 'the': 9}

In [8]:
vocab_size = len(word_to_idx) + 1

In [9]:
vocab_size

10

In [10]:
embedding_dim = 50

In [11]:
class LSTMModel(nn.Module):
    def __init__(self):
        super(LSTMModel, self).__init__()
        self.embedding = nn.Embedding(10, 50)
        self.lstm = nn.LSTM(input_size=50, hidden_size=64, batch_first=True)
        self.fc = nn.Linear(64, 10)

    def forward(self, x):
        # Input x: (batch_size, seq_length)
        x = self.embedding(x) # After embedding: (batch_size, seq_length, embedding_dim)
        output, (hidden, cell) = self.lstm(x) # After LSTM: (batch_size, seq_length, hidden_size)
        # return(output.shape, hidden.shape, cell.shape) # hidden: (1, batch_size, embedding_dim)
        output = output[:, -1, :] # Take the last output of the LSTM: (batch_size, hidden_size
        return self.fc(output) # After linear layer: (batch_size, vocab_size)


In [12]:
model = LSTMModel()

In [13]:
allinput_seq = []
alloutput_seq = []
for sentence in sentences:
    encoded = [word_to_idx[word] for word in sentence]
    for i in range(1, len(encoded)):
        input_seq = encoded[:i]
        target_seq = encoded[i:i+1]
        allinput_seq.append(torch.tensor(input_seq))
        alloutput_seq.append(torch.tensor(target_seq))

In [14]:
allinput_seq

[tensor([9]),
 tensor([9, 2]),
 tensor([4]),
 tensor([4, 1]),
 tensor([9]),
 tensor([9, 3]),
 tensor([9, 3, 5])]

In [26]:
alloutput_seq

[tensor([2]),
 tensor([8]),
 tensor([1]),
 tensor([6]),
 tensor([3]),
 tensor([5]),
 tensor([7])]

In [15]:
from torch.nn.utils.rnn import pad_sequence

In [16]:
inputs = pad_sequence([p for p in allinput_seq], batch_first=True)
targets = pad_sequence([p for p in alloutput_seq], batch_first=True)

In [17]:
inputs

tensor([[9, 0, 0],
        [9, 2, 0],
        [4, 0, 0],
        [4, 1, 0],
        [9, 0, 0],
        [9, 3, 0],
        [9, 3, 5]])

In [18]:
inputs.shape

torch.Size([7, 3])

In [19]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [20]:
out = model(inputs)

In [27]:
out

tensor([[ 1.0371e-01,  1.6113e-02, -1.2736e-01,  1.4140e-01,  9.1141e-02,
         -2.9191e-02,  2.3452e-01, -1.9284e-01,  1.5916e-02,  9.4394e-02],
        [ 1.4289e-01,  5.1716e-02, -1.1060e-01,  1.5547e-01,  1.0779e-02,
          1.3420e-02,  1.8038e-01, -1.5820e-01,  6.2471e-02,  3.4003e-02],
        [ 6.8367e-02,  1.1734e-02, -1.2958e-01,  1.5041e-01,  9.8667e-02,
         -3.4057e-02,  2.4726e-01, -2.0797e-01,  5.7250e-05,  9.1508e-02],
        [ 8.9635e-02, -9.7175e-03, -1.5213e-01,  1.3824e-01,  6.3072e-02,
          1.8166e-02,  1.8090e-01, -1.7916e-01,  1.1211e-02,  1.1106e-02],
        [ 1.0371e-01,  1.6113e-02, -1.2736e-01,  1.4140e-01,  9.1141e-02,
         -2.9191e-02,  2.3452e-01, -1.9284e-01,  1.5916e-02,  9.4394e-02],
        [ 1.3034e-01,  1.1512e-02, -1.6801e-01,  5.6758e-02,  2.4087e-02,
         -4.7924e-02,  2.2575e-01, -1.8634e-01, -1.4626e-02,  9.2621e-02],
        [ 1.6608e-01,  1.3791e-01, -2.2524e-01, -6.2655e-03, -3.1513e-02,
          4.3209e-02,  1.3167e-0

In [21]:
out.shape

torch.Size([7, 10])

In [22]:
out.view(-1, vocab_size).shape

torch.Size([7, 10])

In [23]:
targets.view(-1).shape

torch.Size([7])

In [24]:
EPOCHS = 30
model.train()
for epoch in range(EPOCHS):

    optimizer.zero_grad()

    outputs = model(inputs)
    loss = criterion(outputs.view(-1, vocab_size), targets.view(-1))

    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {loss.item()}")

Epoch 10/30, Loss: 0.36920371651649475
Epoch 20/30, Loss: 0.203123539686203
Epoch 30/30, Loss: 0.19907791912555695


In [25]:
model.eval()
with torch.no_grad():
    input_seq = torch.tensor([word_to_idx["the"], word_to_idx["dog"]]).unsqueeze(0)
    print("Input sequence:", "the", "dog")
    print("Predicted sequence:")
    output = model(input_seq)
    for _ in range(3):
        output = model(input_seq)
        next_token = output[-1:].argmax(dim=-1)
        predicted_word = [word for word, index in word_to_idx.items() if index == next_token.item()][0]
        print(predicted_word)
        input_seq = torch.cat([input_seq, next_token.unsqueeze(0)], dim=1)

Input sequence: the dog
Predicted sequence:
is
running
running
